In [2]:
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

In [3]:
documents_path = Path("../documents")

text_files = list(documents_path.rglob("*.txt"))

print("Total documents:", len(text_files))

for file in text_files:
    print(file)

Total documents: 4
..\documents\alarm_guides\packaging_alarm_investigation.txt
..\documents\incident_reports\packaging_incident_investigation.txt
..\documents\machine_manuals\packaging_machine_operations.txt
..\documents\maintenance_guides\packaging_maintenance_guidelines.txt


In [4]:
documents = []

for file in text_files:
    text = file.read_text(encoding="utf-8")

    documents.append({
        "source": str(file),
        "text": text
    })

print("Documents loaded:", len(documents))

Documents loaded: 4


In [5]:
def create_chunks(text, chunk_size=800, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [6]:
all_chunks = []

for document in documents:

    chunks = create_chunks(document["text"])

    for i, chunk in enumerate(chunks):

        all_chunks.append({
            "id": f"{Path(document['source']).stem}_{i}",
            "text": chunk,
            "source": document["source"],
            "chunk_id": i
        })

print("Total chunks:", len(all_chunks))

Total chunks: 21


In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Packaging_AI_Control_Tower\venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
test_embedding = embedding_model.encode(
    ["high machine downtime"]
)

print(test_embedding.shape)

(1, 384)


In [9]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="../data/chroma_db"
)

print("ChromaDB connected successfully")

ChromaDB connected successfully


In [10]:
collection = chroma_client.get_or_create_collection(
    name="packaging_knowledge"
)

print("Collection created successfully")
print("Current documents:", collection.count())

Collection created successfully
Current documents: 0


In [11]:
texts = [
    chunk["text"]
    for chunk in all_chunks
]

print("Total chunks:", len(texts))

Total chunks: 21


In [12]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (21, 384)


In [13]:
ids = [
    chunk["id"]
    for chunk in all_chunks
]

print("Total IDs:", len(ids))
print(ids[:5])

Total IDs: 21
['packaging_alarm_investigation_0', 'packaging_alarm_investigation_1', 'packaging_alarm_investigation_2', 'packaging_alarm_investigation_3', 'packaging_alarm_investigation_4']


In [14]:
metadatas = [
    {
        "source": chunk["source"],
        "chunk_id": chunk["chunk_id"]
    }
    for chunk in all_chunks
]

print(metadatas[:2])

[{'source': '..\\documents\\alarm_guides\\packaging_alarm_investigation.txt', 'chunk_id': 0}, {'source': '..\\documents\\alarm_guides\\packaging_alarm_investigation.txt', 'chunk_id': 1}]


In [15]:
collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Documents stored successfully")
print("Total documents in ChromaDB:", collection.count())

Documents stored successfully
Total documents in ChromaDB: 21


In [16]:
def search_knowledge(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query]
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    return results

In [17]:
query = "What should I investigate when machine downtime is high?"

results = search_knowledge(
    query,
    top_k=3
)

print("Search completed")

Search completed


In [18]:
for i, document in enumerate(results["documents"][0]):

    print("=" * 70)

    print("RESULT:", i + 1)

    print("SOURCE:")
    print(results["metadatas"][0][i]["source"])

    print("\nCONTENT:")
    print(document)

RESULT: 1
SOURCE:
..\documents\machine_manuals\packaging_machine_operations.txt

CONTENT:
utput and may indicate an operational issue.

When downtime increases, the investigation should consider:

- Frequency of downtime
- Duration of downtime
- Whether downtime is scheduled
- Whether downtime is associated with alarms
- Whether production decreases at the same time
- Historical machine behavior

4. IDLE TIME

Idle time represents periods where the machine is not actively producing.

High idle time can contribute to reduced production.

Possible investigation areas include:

- Production scheduling
- Material availability
- Operator activity
- Machine waiting conditions
- Process transitions

Idle time should not automatically be classified as equipment failure.

5. PERFORMANCE LOSS

Performance loss represents a reduction from expected operational performance.

High performanc
RESULT: 2
SOURCE:
..\documents\maintenance_guides\packaging_maintenance_guidelines.txt

CONTENT:
activity
- P

In [19]:
query = "How should repeated machine alarms be investigated?"

results = search_knowledge(
    query,
    top_k=3
)

for i, document in enumerate(results["documents"][0]):

    print("=" * 70)
    print("RESULT:", i + 1)
    print("SOURCE:", results["metadatas"][0][i]["source"])
    print("\n", document)

RESULT: 1
SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

 c component has failed.

2. ALARM INVESTIGATION

When an alarm is detected, the control tower should examine:

- Equipment ID
- Alarm identifier
- Alarm occurrence time
- Alarm duration when available
- Machine production around the event
- Downtime around the event
- Performance loss around the event
- Previous alarms
- Anomaly detection results
- Risk prediction results

3. RECENT ALARM ANALYSIS

The system should determine whether the alarm is:

- A single isolated event
- Repeated over a short period
- Associated with production reduction
- Associated with increased downtime
- Associated with an anomaly
- Associated with elevated predicted risk

Repeated events may require additional investigation.

4. ALARM FREQUENCY

A high alarm count may indicate repeated operational events.

Howev
RESULT: 2
SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

 estigation.

4. ALARM FREQUENCY

A high

In [20]:
query = "What maintenance actions should be considered for high downtime?"

results = search_knowledge(
    query,
    top_k=3
)

for i, document in enumerate(results["documents"][0]):

    print("=" * 70)
    print("RESULT:", i + 1)
    print("SOURCE:", results["metadatas"][0][i]["source"])
    print("\n", document)

RESULT: 1
SOURCE: ..\documents\maintenance_guides\packaging_maintenance_guidelines.txt

 identify:

- Repeated anomalies
- Repeated alarms
- Repeated downtime
- Repeated production deterioration

Repeated events can be prioritized for human investigation.

6. MAINTENANCE PRIORITY

A project-defined priority can consider:

- Severity of operational deterioration
- Risk prediction
- Anomaly detection
- Production impact
- Downtime impact
- Frequency of abnormal events

Example priority categories:

CRITICAL:
Immediate human investigation recommended.

HIGH:
Maintenance investigation should be prioritized.

MEDIUM:
Monitor and investigate when appropriate.

LOW:
Continue monitoring.

These categories are project decision-support labels and are not manufacturer-defined severity levels.

7. BEFORE MAINTENANCE

Before physical maintenance:

- Verify the machine condition
- Verify t
RESULT: 2
SOURCE: ..\documents\machine_manuals\packaging_machine_operations.txt

 entify unusual behavior and p

In [21]:
def retrieve_context(query, top_k=5):

    results = search_knowledge(
        query,
        top_k=top_k
    )

    context_parts = []

    for i, document in enumerate(
        results["documents"][0]
    ):

        source = results["metadatas"][0][i]["source"]

        context_parts.append(
            f"""
SOURCE: {source}

CONTENT:
{document}
"""
        )

    return "\n\n".join(context_parts)

In [22]:
query = """
Machine has high downtime and repeated alarms.
What should be investigated?
"""

context = retrieve_context(
    query,
    top_k=5
)

print(context)


SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

CONTENT:
c component has failed.

2. ALARM INVESTIGATION

When an alarm is detected, the control tower should examine:

- Equipment ID
- Alarm identifier
- Alarm occurrence time
- Alarm duration when available
- Machine production around the event
- Downtime around the event
- Performance loss around the event
- Previous alarms
- Anomaly detection results
- Risk prediction results

3. RECENT ALARM ANALYSIS

The system should determine whether the alarm is:

- A single isolated event
- Repeated over a short period
- Associated with production reduction
- Associated with increased downtime
- Associated with an anomaly
- Associated with elevated predicted risk

Repeated events may require additional investigation.

4. ALARM FREQUENCY

A high alarm count may indicate repeated operational events.

Howev



SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

CONTENT:
estigation.

4. ALARM FREQUENCY

A high

In [24]:
from pathlib import Path
import os

print("Current notebook folder:")
print(Path.cwd())

print("\nProject root detected:")
project_root = Path.cwd().parent
print(project_root)

print("\n.env path:")
env_file = project_root / ".env"
print(env_file)

print("\nDoes .env exist?")
print(env_file.exists())

Current notebook folder:
C:\Packaging_AI_Control_Tower\notebooks

Project root detected:
C:\Packaging_AI_Control_Tower

.env path:
C:\Packaging_AI_Control_Tower\.env

Does .env exist?
True


In [27]:
from dotenv import load_dotenv
import os

load_dotenv(env_file)

api_key = os.getenv("GEMINI_API_KEY")

print("Key found:", api_key is not None)
print("Key length:", len(api_key) if api_key else 0)

Key found: True
Key length: 53


In [28]:
import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
env_file = project_root / ".env"

load_dotenv(env_file)

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("Gemini API key loaded successfully")
else:
    print("Gemini API key NOT found")

Gemini API key loaded successfully


In [29]:
from google import genai

client = genai.Client(
    api_key=api_key
)

print("Gemini client created successfully")

Gemini client created successfully


In [30]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Explain machine downtime in a packaging industry in simple words."
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In the packaging industry, **machine downtime** is simply any period when your packaging machines are **not running** and are not producing finished goods.

Think of it like a car that is parked in the garage instead of driving on the road. When the machine is "down," the factory isn’t making any money.

Here is a breakdown of why it happens and why it matters:

### 1. What counts as downtime?
Downtime generally falls into two categories:

*   **Planned Downtime (The "Good" Kind):** This is scheduled. You know it’s coming, so you can prepare for it. Examples include:
    *   **Maintenance:** Cleaning or oiling the machines to keep them in good shape.
    *   **Changeovers:** Adjusting the machine to switch from packaging a small box to a large box.
    *   **Shift Breaks:** Stopping the machine because the operators are on lunch.
*   **Unplanned Downtime (The "Bad" Kind):** This is unexpected and usually causes stress. Examples include:
    *   **Breakdowns:** A part breaks, a sensor f

In [31]:
user_query = """
Machine has high downtime and repeated alarms.
What should be investigated?
"""

In [32]:
context = retrieve_context(
    user_query,
    top_k=5
)

print(context)


SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

CONTENT:
c component has failed.

2. ALARM INVESTIGATION

When an alarm is detected, the control tower should examine:

- Equipment ID
- Alarm identifier
- Alarm occurrence time
- Alarm duration when available
- Machine production around the event
- Downtime around the event
- Performance loss around the event
- Previous alarms
- Anomaly detection results
- Risk prediction results

3. RECENT ALARM ANALYSIS

The system should determine whether the alarm is:

- A single isolated event
- Repeated over a short period
- Associated with production reduction
- Associated with increased downtime
- Associated with an anomaly
- Associated with elevated predicted risk

Repeated events may require additional investigation.

4. ALARM FREQUENCY

A high alarm count may indicate repeated operational events.

Howev



SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

CONTENT:
estigation.

4. ALARM FREQUENCY

A high

In [33]:
prompt = f"""
You are an AI Operations Assistant for a
Packaging Industry Operations Control Tower.

Your task is to analyze operational problems using
the retrieved project knowledge.

USER QUESTION:
{user_query}

RETRIEVED KNOWLEDGE:
{context}

INSTRUCTIONS:

1. Use the retrieved knowledge to answer the question.

2. Clearly separate observed facts from possible causes.

3. Do not invent the meaning of any alarm code.

4. If the retrieved knowledge does not provide
   enough information, say:
   "Insufficient information in the knowledge base."

5. Do not claim that a possible cause is a confirmed
   root cause.

6. Provide practical investigation steps.

7. Physical maintenance or machine intervention
   requires human verification.

Return the answer using this format:

Incident Summary:
Evidence:
Possible Root Cause:
Confidence:
Recommended Investigation:
Human Verification:
"""

In [34]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

print(response.text)

Incident Summary:
The machine is experiencing high levels of downtime coupled with recurring alarms, indicating a potential degradation in operational performance.

Evidence:
*   High frequency of alarm events.
*   Increased machine downtime.
*   Possible correlation between alarm timing, downtime duration, and production output (based on current operational logs).

Possible Root Cause:
*   **Operational/Process Issues:** Recurring alarms may stem from material availability issues, operator activity, or process transitions leading to idle time.
*   **Equipment Performance:** The machine may be experiencing performance loss due to mechanical wear or recent changes, resulting in repeated interruptions.
*   **Systemic Deterioration:** The combination of repeated anomalies, alarms, and production drops may indicate a broader underlying equipment fault or calibration drift.

Confidence:
Moderate (Correlation between alarms and downtime requires further cross-referencing with machine history

In [35]:
def ask_rag_gemini(user_query, top_k=5):

    # Step 1: Retrieve relevant knowledge
    context = retrieve_context(
        user_query,
        top_k=top_k
    )

    # Step 2: Create prompt
    prompt = f"""
You are an AI Operations Assistant for a
Packaging Industry Operations Control Tower.

USER QUESTION:
{user_query}

RETRIEVED KNOWLEDGE:
{context}

RULES:

- Use retrieved knowledge as the primary
  project-specific source.
- Do not invent alarm meanings.
- Separate facts from possible causes.
- Do not present possible causes as confirmed causes.
- If information is unavailable, say so.
- Recommend investigation steps.
- Human verification is required before
  physical intervention.

Return:

Incident Summary:
Evidence:
Possible Root Cause:
Confidence:
Recommended Investigation:
Human Verification:
"""

    # Step 3: Ask Gemini
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return response.text

In [36]:
answer = ask_rag_gemini(
    "Machine has high downtime and repeated alarms. What should be investigated?"
)

print(answer)

**Incident Summary:**
The machine is experiencing high downtime and repeated alarms, which may indicate operational deterioration or underlying equipment issues.

**Evidence:**
*   **Alarm Frequency:** Repeated alarms occurring over a short period.
*   **Downtime Patterns:** Increased downtime associated with these alarms.
*   **Performance Indicators:** Potential correlation between alarm events, downtime, and production loss.
*   **Operational Context:** Historical machine behavior and any recent changes to the machine or operational processes.

**Possible Root Cause:**
The observed downtime and alarms may be associated with operational deterioration, performance loss, or specific mechanical/system malfunctions. *Note: The system cannot confirm these as causes without further integrated data analysis.*

**Confidence:**
Low to Moderate (Evidence suggests an issue, but the specific root cause requires further cross-referencing of operational data).

**Recommended Investigation:**
To de